<a href="https://colab.research.google.com/github/a-i-git/Deep-Learning/blob/main/ANN_MNIST_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Moving the computation from the CPU to T4-GPU

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [2]:
#Change 1
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device {device}')

using device cuda


In [3]:
df=pd.read_csv('/content/fashion-mnist_train.csv')
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=['label'],axis=1)
y=df.iloc[:,0].values
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

In [6]:
from torch.utils.data import Dataset,DataLoader
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [7]:
train_dataset=CustomDataset(X_train_scaled,y_train)

In [8]:
test_dataset=CustomDataset(X_test_scaled,y_test)

Mini-Batch GD

In [9]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

#Pin_memory - accelerated the data transfer from CPU to GPU

NN Model

In [10]:
import torch.nn as nn
class NNModel(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10),
        # nn.Softmax()
    )
  def forward(self,x):
    return self.network(x)


In [11]:
learning_rate=0.1
epochs=25

In [12]:
import torch.optim as optim
model=NNModel(X_train.shape[1])
#Change 2 - Move the model to GPU(device)
model=model.to(device)

criterion=nn.CrossEntropyLoss()

optimiser=optim.SGD(model.parameters(),lr=learning_rate)

#Training Loop

In [15]:
for epoch in range(epochs):
  total_epoch_loss=0

  for batch_features,batch_labels in train_loader:

    #Change 3- Put the features and labels in GPU
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)


    outputs=model(batch_features) #Forward Pass

    loss=criterion(outputs,batch_labels) #Calculate Loss

    #Backpropagation
    optimiser.zero_grad()
    loss.backward()

    #Update weights
    optimiser.step()

    total_epoch_loss=total_epoch_loss+loss.item()

  avg_loader=total_epoch_loss/len(train_loader)
  print(f'Epoch {epoch} Loss:{avg_loader}')

Epoch 0 Loss:0.49307713799675307
Epoch 1 Loss:0.3621536966015895
Epoch 2 Loss:0.31708121377478043
Epoch 3 Loss:0.28904386918246744
Epoch 4 Loss:0.2694949116917948
Epoch 5 Loss:0.24937950420131286
Epoch 6 Loss:0.23465496184676884
Epoch 7 Loss:0.21889649140338102
Epoch 8 Loss:0.2064559128768742
Epoch 9 Loss:0.19843352464338143
Epoch 10 Loss:0.19069561354226122
Epoch 11 Loss:0.17812982477930686
Epoch 12 Loss:0.17169141756991546
Epoch 13 Loss:0.16110490964911878
Epoch 14 Loss:0.1532812358768036
Epoch 15 Loss:0.14321603925464055
Epoch 16 Loss:0.14381040971819312
Epoch 17 Loss:0.13849375276081263
Epoch 18 Loss:0.13001988939366613
Epoch 19 Loss:0.12289927476244823
Epoch 20 Loss:0.1219554350693555
Epoch 21 Loss:0.1151018347859693
Epoch 22 Loss:0.11463475858210587
Epoch 23 Loss:0.11093854026426561
Epoch 24 Loss:0.10534454192360863


Evaluation Code

In [16]:
model.eval()

NNModel(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

#Testing/Prediction Loop

In [17]:
total=0
correct=0

with torch.no_grad():# No gradients to be calculated while prediction
  for batch_features,batch_labels in test_loader:
    #Change-4 : Same as Training Loop
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)


    outputs=model(batch_features)

    # print(outputs)  # gives the probabilites of all the 10 classes

    _,predicted=torch.max(outputs,1) # Taking the maximum of all the probabilities

    total=total+batch_labels.shape[0]

    correct=correct+(predicted==batch_labels).sum().item()

print(correct/total)

0.8803333333333333
